# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidism/Machine-Learning-intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata
import pandas as pd
import numpy as np
import datetime

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="dim_content.parquet", repo_type="dataset")
perf_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset")

dim_content = pd.read_parquet(content_path)
fact_perf = pd.read_parquet(perf_path)
fact_perf["report_date"] = pd.to_datetime(fact_perf["report_date"])

available = fact_perf[(fact_perf["gsc_data_available"] == True) & (fact_perf["ga4_data_available"] == True)]

page_agg = available.groupby(["content_hash_id","client_hash_id"]).agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean")
).reset_index()
page_agg["ctr"] = page_agg["total_clicks"] / page_agg["total_impressions"].replace(0, pd.NA)

df = page_agg.merge(dim_content[["content_hash_id","word_count","char_count","backlinks","search_volume"]],
                     on="content_hash_id", how="left")
df = df.dropna(subset=["avg_position","ctr","word_count","char_count","backlinks","search_volume"])
df = df[df["total_impressions"] >= 50]  # same impression floor lesson from Week 4

# Position-tier expected CTR gap (same signal that worked in Week 4)
df["position_bucket"] = pd.cut(df["avg_position"], bins=[0,3,10,20,50,1000], labels=["1-3","4-10","11-20","21-50","50+"])
df["expected_ctr"] = df.groupby("position_bucket", observed=True)["ctr"].transform("mean")
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

# Label: declining = below-median CTR performance for its position tier (same logic as baseline signal)
first_half = fact_perf[fact_perf["report_date"] < "2026-03-16"]
second_half = fact_perf[fact_perf["report_date"] >= "2026-03-16"]
imp_first = first_half.groupby("content_hash_id")["gsc_impressions"].sum()
imp_second = second_half.groupby("content_hash_id")["gsc_impressions"].sum()
trend = pd.DataFrame({"imp_first": imp_first, "imp_second": imp_second}).dropna()
trend["label_down"] = (trend["imp_second"] < trend["imp_first"]).astype(int)

df = df.merge(trend[["label_down"]], left_on="content_hash_id", right_index=True, how="inner")
print("Final modeling dataset:", df.shape)
print("Label balance:\n", df["label_down"].value_counts(normalize=True))

Final modeling dataset: (26562, 14)
Label balance:
 label_down
0    0.630412
1    0.369588
Name: proportion, dtype: float64


## 1. Method choice and why
**Method: Random Forest Classifier**

Random Forest fits this lane because the earlier starter notebooks showed a clear ~3x lift
over hand-written rules (0.240 -> 0.740 Precision@50) on this same kind of tabular, mixed-signal
problem. My features (word_count, char_count, backlinks, avg_position, ctr_gap, search_volume)
have non-linear relationships and interactions with the label — e.g., ctr_gap matters
differently depending on position tier — which a tree ensemble captures naturally without
manual interaction terms. I chose Random Forest over Gradient Boosting for this pass because
it's more robust to the imbalanced, noisy label I'm working with, and easier to interpret via
feature_importances_ for the error analysis in Section 4.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

Split: Grouped by client_hash_id (GroupShuffleSplit), 70/30

A random row-level split would let pages from the same client appear in both train and test,
letting the model implicitly memorize client-specific patterns rather than learning
generalizable content signals. Since the real decision this supports needs to generalize to
clients the reviewer hasn't seen scored yet, a client-grouped split is the honest way to
estimate real-world performance, matching the client-holdout approach used in the reference
pipeline (scripts/03_train_model.py).

In [7]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["word_count","char_count","backlinks","avg_position","ctr_gap","search_volume"]
X = df[feature_cols]
y = df["label_down"]
groups = df["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print("Train rows:", len(X_train), " Test rows:", len(X_test))
print("Client overlap between train and test:", len(train_clients & test_clients), "(should be 0)")

Train rows: 11649  Test rows: 14913
Client overlap between train and test: 0 (should be 0)


## 3. Train + compare vs my baseline

Comparison table above shows Precision@20/50/100 for the Week-4 rule-based baseline (ctr_gap
only) vs a Random Forest model, evaluated on the same client-grouped test set. The model
outperforms the baseline at every k — most notably at k=100, where precision jumps from 0.43
to 0.70. This confirms the additional features (word_count, char_count, backlinks,
search_volume) carry real predictive signal beyond ctr_gap alone, and that signal generalizes
to clients unseen during training, since the split was client-grouped. The gap is largest at
k=100, suggesting the model is especially better than the baseline at surfacing good
candidates further down a longer review queue — not just at the very top.

In [8]:
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    return y_true.values[top_k].mean()

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# Baseline: same ctr_gap-based score from Week 4, evaluated on the SAME test set
baseline_scores = X_test["ctr_gap"].clip(lower=0).values

results = []
for k in (20, 50, 100):
    results.append({
        "k": k,
        "baseline_precision": round(precision_at_k(baseline_scores, y_test, k), 3),
        "random_forest_precision": round(precision_at_k(rf_scores, y_test, k), 3)
    })

results_df = pd.DataFrame(results)
print(results_df)

     k  baseline_precision  random_forest_precision
0   20                0.40                     0.60
1   50                0.48                     0.64
2  100                0.43                     0.70


## 4. Errors and interpretation

Feature importances show char_count (0.26) and word_count (0.25) as the strongest predictors
— together explaining nearly half the model's decisions — followed by avg_position (0.18) and
ctr_gap (0.13). search_volume and backlinks contribute least (0.10 and 0.08). This suggests
content depth/length signals carry more weight than I expected going in, more than the CTR-vs-
position gap that anchored my Week-4 baseline.

Looking at the declining pages the model ranked lowest (missed opportunities): they share a
clear pattern — very strong avg_position (mostly under 3, meaning they're already ranking #1-2),
near-zero ctr_gap (CTR looks fine for their position), low search_volume (0-30), and mostly
zero backlinks. The model is essentially trusting "already ranks well + CTR looks fine" as a
signal of health, and deprioritizing these pages because none of my 6 features flag a problem —
even though they are genuinely declining by the label. This points to a real blind spot: pages
that are declining for reasons my feature set doesn't capture (e.g., a competitor overtaking
them slightly, or a slow multi-month erosion not visible in a single position/CTR snapshot)
get systematically under-ranked. A reviewer relying on this model alone would miss these — a
useful addition for a future iteration would be a trend-based feature (e.g., week-over-week
position change within March) rather than only point-in-time snapshots.

In [9]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:\n", importances)

test_results = X_test.copy()
test_results["y_true"] = y_test.values
test_results["rf_score"] = rf_scores
test_results = test_results.sort_values("rf_score", ascending=False).reset_index(drop=True)

missed = test_results[(test_results["y_true"]==1)].tail(10)
print("\nSample of declining pages the model ranked LOW (missed opportunities):")
print(missed)

Feature importances:
 char_count       0.262404
word_count       0.248925
avg_position     0.182533
ctr_gap          0.127694
search_volume    0.100576
backlinks        0.077868
dtype: float64

Sample of declining pages the model ranked LOW (missed opportunities):
       word_count  char_count  backlinks  avg_position   ctr_gap  \
14860      2767.0     19113.0        0.0      1.338651  0.004720   
14865      2549.0     18818.0        0.0      2.571233  0.010690   
14866      2815.0     19084.0       22.0      1.640185  0.008367   
14873      3143.0     22460.0      283.0      1.732143  0.010690   
14875      2696.0     18409.0        0.0      1.316236  0.007384   
14876      2799.0     18563.0        0.0      1.223543  0.001315   
14883      2628.0     19159.0        0.0      1.901567  0.004648   
14906      2522.0     18979.0        0.0      1.922156  0.010690   
14909      2767.0     19181.0        0.0      0.934491  0.008145   
14912      2480.0     18406.0       89.0      1.868625 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.